In [1]:
import os
import re
import time
import pickle
import numpy as np
import pandas as pd
from tqdm import tqdm
from rank_bm25 import BM25Okapi
import nltk
from nltk.stem import PorterStemmer
from nltk.corpus import stopwords

# Khai bao duong dan
PROCESSED_DIR = "../data/processed"
RAW_DIR = "../data/raw"
MODELS_DIR = "../models"
os.makedirs(MODELS_DIR, exist_ok=True)

# Cau hinh tien xu ly
stemmer = PorterStemmer()
STANDARD_STOP = set(stopwords.words('english'))
CUSTOM_STOP_STEMMED = {
    'paper', 'propos', 'method', 'approach', 'result',
    'show', 'base', 'also', 'howev', 'therefor', 'thu',
    'furthermor', 'present', 'work', 'studi', 'experiment', 'evalu'
}
PATTERN_CLEAN = re.compile(r'[^a-z0-9\s]')
PATTERN_SPACE = re.compile(r'\s+')

def preprocess_query(text: str) -> list:
    text = str(text).lower()
    text = PATTERN_CLEAN.sub(' ', text)
    text = PATTERN_SPACE.sub(' ', text).strip()
    tokens = [t for t in text.split() if t not in STANDARD_STOP and len(t) > 2]
    stemmed = [stemmer.stem(t) for t in tokens]
    return [t for t in stemmed if t not in CUSTOM_STOP_STEMMED]

In [2]:
print("Dang nap du lieu...")

corpus_df = pd.read_parquet(f"{PROCESSED_DIR}/corpus_bm25.parquet")
doc_ids = corpus_df['id'].astype(str).tolist()

queries_df = pd.read_parquet(f"{RAW_DIR}/queries.parquet")
queries_dict = dict(zip(queries_df['_id'].astype(str), queries_df['text']))

qrels_df = pd.read_parquet(f"{RAW_DIR}/qrels.parquet")
qrels_dict = {}
for _, row in qrels_df.iterrows():
    qid = str(row['query-id'])
    did = str(row['corpus-id'])
    score = int(row['score'])
    if qid not in qrels_dict:
        qrels_dict[qid] = {}
    qrels_dict[qid][did] = score

print(f"Corpus: {len(doc_ids)} tai lieu")
print(f"Queries: {len(queries_dict)} cau")
print(f"Qrels: {len(qrels_dict)} cau co dap an")

Dang nap du lieu...
Corpus: 25657 tai lieu
Queries: 1000 cau
Qrels: 1000 cau co dap an


In [3]:
import math

def precision_at_k(retrieved, relevant, k):
    hits = sum(1 for d in retrieved[:k] if d in relevant and relevant[d] > 0)
    return hits / k

def recall_at_k(retrieved, relevant, k):
    total_rel = sum(1 for v in relevant.values() if v > 0)
    if total_rel == 0: return 0.0
    hits = sum(1 for d in retrieved[:k] if d in relevant and relevant[d] > 0)
    return hits / total_rel

def ndcg_at_k(retrieved, relevant, k):
    def dcg(ranked_list):
        return sum(relevant.get(d, 0) / math.log2(i + 2) for i, d in enumerate(ranked_list[:k]))
    actual_dcg = dcg(retrieved)
    ideal_list = sorted(relevant.keys(), key=lambda x: relevant[x], reverse=True)
    ideal_dcg = dcg(ideal_list)
    return actual_dcg / ideal_dcg if ideal_dcg > 0 else 0.0

def average_precision(retrieved, relevant):
    hits = 0
    ap = 0.0
    total_rel = sum(1 for v in relevant.values() if v > 0)
    if total_rel == 0: return 0.0
    for i, d in enumerate(retrieved):
        if d in relevant and relevant[d] > 0:
            hits += 1
            ap += hits / (i + 1)
    return ap / total_rel

In [4]:
print("Dang tach tu Corpus...")
tokenized_corpus = [str(text).split() for text in corpus_df['text_bm25']]

# Luoi tham so rut gon de chay nhanh hon
k1_values = [1.0, 1.2, 1.5]
b_values = [0.5, 0.75, 0.9]

best_map = 0.0
best_params = {'k1': 1.5, 'b': 0.75}
valid_qids = [qid for qid in qrels_dict.keys() if qid in queries_dict]

print("Dang tien hanh Grid Search (khoang 10-15 phut)...")
for k1 in k1_values:
    for b in b_values:
        bm25_test = BM25Okapi(tokenized_corpus, k1=k1, b=b)
        map_scores = []
        
        for qid in valid_qids:
            query_text = queries_dict[qid]
            relevant_docs = qrels_dict[qid]
            tokenized_query = preprocess_query(query_text)
            scores = bm25_test.get_scores(tokenized_query)
            
            top_indices = np.argsort(scores)[::-1][:20]
            retrieved_docs = [doc_ids[idx] for idx in top_indices]
            map_scores.append(average_precision(retrieved_docs, relevant_docs))
            
        current_map = np.mean(map_scores)
        print(f"[k1={k1:<3} | b={b:<4}] MAP: {current_map:.4f}")
        
        if current_map > best_map:
            best_map = current_map
            best_params = {'k1': k1, 'b': b}

print("-" * 40)
print(f"Bo tham so tot nhat: k1={best_params['k1']}, b={best_params['b']}")

Dang tach tu Corpus...
Dang tien hanh Grid Search (khoang 10-15 phut)...
[k1=1.0 | b=0.5 ] MAP: 0.0981
[k1=1.0 | b=0.75] MAP: 0.0998
[k1=1.0 | b=0.9 ] MAP: 0.1004
[k1=1.2 | b=0.5 ] MAP: 0.0993
[k1=1.2 | b=0.75] MAP: 0.1015
[k1=1.2 | b=0.9 ] MAP: 0.1023
[k1=1.5 | b=0.5 ] MAP: 0.1006
[k1=1.5 | b=0.75] MAP: 0.1024
[k1=1.5 | b=0.9 ] MAP: 0.1033
----------------------------------------
Bo tham so tot nhat: k1=1.5, b=0.9


In [5]:
print("Dang huan luyen mo hinh Final voi bo tham so tot nhat...")
bm25_final = BM25Okapi(tokenized_corpus, k1=best_params['k1'], b=best_params['b'])

# Luu model dang Pickle cho Web Demo load vao RAM
with open(f"{MODELS_DIR}/bm25_model.pkl", "wb") as f:
    pickle.dump(bm25_final, f)

# Luu danh sach doc_ids dang CSV
df_doc_ids = pd.DataFrame({'doc_id': doc_ids})
df_doc_ids.to_csv(f"{MODELS_DIR}/bm25_doc_ids.csv", index=False)

print(f"Da luu bm25_model.pkl va bm25_doc_ids.csv tai {MODELS_DIR}")

Dang huan luyen mo hinh Final voi bo tham so tot nhat...
Da luu bm25_model.pkl va bm25_doc_ids.csv tai ../models


In [6]:
k_values = [5, 10, 20]
results = {k: {'P': [], 'R': [], 'NDCG': []} for k in k_values}
latencies = []

print("Dang danh gia chi tiet Final Model tren tap Test...")
for qid in tqdm(valid_qids):
    query_text = queries_dict[qid]
    relevant_docs = qrels_dict[qid]
    
    t0 = time.time()
    tokenized_query = preprocess_query(query_text)
    
    scores = bm25_final.get_scores(tokenized_query)
    top_indices = np.argsort(scores)[::-1][:max(k_values)]
    retrieved_docs = [doc_ids[idx] for idx in top_indices]
    
    latencies.append(time.time() - t0)
    
    for k in k_values:
        results[k]['P'].append(precision_at_k(retrieved_docs, relevant_docs, k))
        results[k]['R'].append(recall_at_k(retrieved_docs, relevant_docs, k))
        results[k]['NDCG'].append(ndcg_at_k(retrieved_docs, relevant_docs, k))

# Chuyen doi ket qua thanh DataFrame de luu CSV
eval_data = []
for k in k_values:
    eval_data.append({
        'Top_K': k,
        'Precision': round(np.mean(results[k]['P']), 4),
        'Recall': round(np.mean(results[k]['R']), 4),
        'NDCG': round(np.mean(results[k]['NDCG']), 4)
    })
    
df_eval = pd.DataFrame(eval_data)
df_eval.to_csv(f"{MODELS_DIR}/bm25_eval_metrics.csv", index=False)

print("\n=== KET QUA DANH GIA FINAL ===")
print(f"Thoi gian phan hoi trung binh: {np.mean(latencies)*1000:.2f} ms / cau")
print(df_eval.to_string(index=False))
print(f"\nDa luu bang chi so danh gia vao: {MODELS_DIR}/bm25_eval_metrics.csv")

Dang danh gia chi tiet Final Model tren tap Test...


100%|██████████| 1000/1000 [01:00<00:00, 16.44it/s]



=== KET QUA DANH GIA FINAL ===
Thoi gian phan hoi trung binh: 60.41 ms / cau
 Top_K  Precision  Recall   NDCG
     5     0.1184  0.1200 0.1355
    10     0.0823  0.1670 0.1611
    20     0.0552  0.2238 0.1855

Da luu bang chi so danh gia vao: ../models/bm25_eval_metrics.csv
